In [1]:
import sys, os
from pathlib import Path
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'modelos').exists())
sys.path.insert(0, str(raiz)); sys.path.insert(0, str(raiz / 'src' / 'dados'))
os.chdir(raiz / 'notebooks' / 'modelos')   # mantém os '../../' do resto funcionando
import numpy as np, pandas as pd, joblib
from scipy.optimize import brentq
from src.modelos.dados import ler_base, preparar
from src.modelos.variaveis import criar_variaveis
from src.modelos.treino import _para_modelo
from src.modelos.prever import juro_justo, LGD_POR_PAIS, LGD_PADRAO

pac = joblib.load('../../modelos_salvos/pacote_prever_v1.joblib')
base = criar_variaveis(preparar(ler_base()), referencia=pac['referencia'])
base = base[~base['fora_da_auditoria']]          # sem reestruturados

base['pd'] = pac['modelo'].predict_proba(_para_modelo(base))[:, 1]
mes = pd.to_datetime(base['data_emprestimo']).dt.to_period('M').dt.to_timestamp()
base['euribor'] = (mes.map(pac['euribor_12m']).fillna(0) / 100).clip(lower=0)
base['lgd'] = base['pais'].map(LGD_POR_PAIS).fillna(LGD_PADRAO)
base['cobrado'] = base['juro_aa'] / 100

def media_justo(d, m):
    return np.mean([juro_justo(int(r.prazo_meses), r.pd, r.lgd, r.euribor + m)
                    for r in d.itertuples()])

margens = {}
linhas = []
linhas = []
for ano, d in base.groupby('ano'):
    d = d.sample(min(len(d), 1000), random_state=0)
    alvo = d['cobrado'].mean()
    f = lambda m: media_justo(d, m) - alvo
    grade = {f'justo_m{int(m*100)}%': round(f(m) + alvo, 3) for m in (-0.05, 0.0, 0.05, 0.10)}
    m = brentq(f, 0.0, 1.0) if f(0.0) < 0 < f(1.0) else None
    linhas.append({'ano': ano, 'cobrado': round(alvo, 3), 'pd': round(d['pd'].mean(), 3),
                   **grade, 'margem': None if m is None else round(m, 4)})
print(pd.DataFrame(linhas).to_string(index=False))




c:\dev\Auditor-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 ano  cobrado    pd  justo_m-5%  justo_m0%  justo_m5%  justo_m10%  margem
2009    0.293 0.370       0.111      0.163      0.213       0.261  0.1347
2010    0.325 0.345       0.088      0.140      0.190       0.238  0.1968
2011    0.255 0.228       0.137      0.079      0.128       0.175  0.1915
2012    0.238 0.217       2.024      0.060      0.109       0.156  0.1927
2013    0.263 0.344       2.288      0.071      0.120       0.167  0.2078
2014    0.293 0.526       1.278      0.188      0.238       0.286  0.1076
2015    0.345 0.638       0.979      0.199      0.249       0.297  0.1527
2016    0.468 0.546       1.258      0.183      0.233       0.281  0.3166
